# LeetCode 2549 — Count Distinct Numbers on Board bằng Python

Link đề: https://leetcode.com/problems/count-distinct-numbers-on-board/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def distinctIntegers(self, n: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy trình "mỗi ngày, từ mỗi số `x` trên bảng thêm các số `i` thỏa `x % i == 1`" và nhận ra rằng sau một số ngày đủ lớn bảng **ổn định** (không thêm số mới).
2. Biết dùng `set` để lưu các số khác nhau, và dùng **mô phỏng đến khi ổn định** thay vì mô phỏng `10^9` ngày.
3. Hiểu quan hệ `x % i == 1` ⇔ `i` là ước của `x - 1` (với `i >= 2`) và từ đó chứng minh đáp án là `n - 1` (khi `n >= 2`).
4. Viết 4 cách: mô phỏng từng ngày đến khi ổn định, BFS với hàng đợi, lan truyền theo ước của `x - 1` từ lớn xuống nhỏ, và công thức `max(1, n - 1)`; đối chiếu bằng test.
5. Tránh các lỗi: chỉ mô phỏng một ngày, trả về `n`, trả về `n - 1` cho `n = 1`, và nhầm điều kiện `x % i == 1` với `x % i == 0`.

## 2. Đề bài & ví dụ

Cho số nguyên dương `n`. Ban đầu trên bảng có đúng **một** số `n`. Mỗi **ngày** (trong `10^9` ngày), với **mỗi số `x`** đang có trên bảng, ta thêm vào bảng mọi số nguyên `i` với `1 <= i <= n` thỏa `x % i == 1`. (Số nào đã có thì không thêm nữa.)

Trả về số lượng số nguyên **khác nhau** trên bảng sau `10^9` ngày.

| Ví dụ | n | Output | Giải thích |
|---|---|---|---|
| 1 | `5` | `4` | bảng cuối cùng gồm `2, 3, 4, 5` |
| 2 | `3` | `2` | `3 % 2 = 1` nên thêm `2`; bảng gồm `2, 3` |

Constraints: `1 <= n <= 100`.

In [1]:
n = 5
board = {n}
day = 0
while True:
    new = {i for x in board for i in range(1, n + 1) if x % i == 1}
    if new <= board:
        break
    board |= new
    day += 1
    print(f"ngày {day}: bảng = {sorted(board)}")
print("n =", n, "-> số khác nhau:", len(board), "(mong đợi 4)")

ngày 1: bảng = [2, 4, 5]
ngày 2: bảng = [2, 3, 4, 5]
n = 5 -> số khác nhau: 4 (mong đợi 4)


## 3. Kiến thức nền

### 3.1. `set` để lưu số khác nhau

`set` chỉ giữ mỗi giá trị một lần. `board.add(x)` thêm một phần tử; `board |= other` hợp hai tập; `a <= b` kiểm tra `a` là tập con của `b`.

In [2]:
board = {5}
board.add(2)
board.add(2)                 # thêm lần nữa vẫn chỉ một phần tử
print(board, len(board))
print({2} <= board, {2, 9} <= board)
board |= {3, 4}
print(sorted(board))

{2, 5} 2
True False
[2, 3, 4, 5]


### 3.2. Điều kiện `x % i == 1`

Phép `%` cho số dư. `x % i == 1` nghĩa là chia `x` cho `i` dư đúng `1`. Với `i = 1` thì `x % 1` luôn bằng `0`, nên **`i = 1` không bao giờ được thêm**.

In [3]:
x = 5
for i in range(1, 6):
    print(f"{x} % {i} = {x % i}", "<- thêm i" if x % i == 1 else "")

5 % 1 = 0 
5 % 2 = 1 <- thêm i
5 % 3 = 2 
5 % 4 = 1 <- thêm i
5 % 5 = 0 


### 3.3. `x % i == 1` ⇔ `i` là ước của `x - 1`

Nếu `x % i == 1` thì `x = k * i + 1`, tức `x - 1 = k * i`, nên `i` chia hết `x - 1`. Ngược lại, nếu `i | (x - 1)` và `i >= 2` thì `x = k * i + 1` có dư `1` (vì `1 < i`). Với `i = 1`, `x - 1` chia hết cho `1` nhưng `x % 1 = 0`, nên phải loại `i = 1`.

In [4]:
x = 12
uoc = [i for i in range(1, x) if (x - 1) % i == 0]
print("ước của", x - 1, ":", uoc)
print("i thỏa x % i == 1:", [i for i in range(1, x + 1) if x % i == 1])

ước của 11 : [1, 11]
i thỏa x % i == 1: [11]


### 3.4. Hàng đợi `deque` (BFS)

`collections.deque` cho `append` và `popleft` đều `O(1)`. BFS xử lý lần lượt từng số trên bảng và đẩy các số mới vào cuối hàng đợi; mỗi số chỉ được xử lý một lần.

In [5]:
from collections import deque
q = deque([5])
q.append(2)
print(q.popleft(), list(q))

5 [2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `n = 5`:

- Ngày 1: bảng `{5}`. `5 % 2 = 1`, `5 % 4 = 1` nên thêm `2` và `4`. Bảng `{5, 2, 4}`.
- Ngày 2: từ `2`: `2 % 1 = 0`, `2 % 2 = 0` không thêm gì. Từ `4`: `4 % 3 = 1` nên thêm `3`. Bảng `{5, 4, 3, 2}`.
- Ngày 3: từ `3`: `3 % 2 = 1` (đã có). Không thay đổi nữa.

Bảng ổn định với `4` số. Không cần mô phỏng `10^9` ngày: khi một ngày **không thêm số mới**, mọi ngày sau cũng thế.

Nhận xét quan trọng: với `n >= 3`, `n % (n - 1) = 1` nên `n - 1` luôn được thêm ngay ngày đầu; rồi `(n - 1) % (n - 2) = 1` (khi `n - 1 >= 3`) nên `n - 2` được thêm, v.v. Cứ thế bảng chứa mọi số từ `2` đến `n`.

In [6]:
for n in (1, 2, 3, 4, 5, 6):
    board, changed = {n}, True
    while changed:
        new = {i for x in board for i in range(1, n + 1) if x % i == 1}
        changed = not new <= board
        board |= new
    print(f"n={n}: bảng cuối = {sorted(board)} -> {len(board)} số")

n=1: bảng cuối = [1] -> 1 số
n=2: bảng cuối = [2] -> 1 số
n=3: bảng cuối = [2, 3] -> 2 số
n=4: bảng cuối = [2, 3, 4] -> 3 số
n=5: bảng cuối = [2, 3, 4, 5] -> 4 số
n=6: bảng cuối = [2, 3, 4, 5, 6] -> 5 số


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng ngày đến khi ổn định

**Ý tưởng:** lặp từng ngày; mỗi ngày với mọi `x` trên bảng, mọi `i` từ `1` đến `n`, nếu `x % i == 1` thì thêm `i`. Dừng khi một ngày không thêm được số nào (bảng ổn định). Không thể chạy thật `10^9` ngày nhưng ta chứng minh rằng bảng không đổi nữa sau ngày ổn định đầu tiên.

In [7]:
class SolutionSimulate:
    def distinctIntegers(self, n: int) -> int:
        board = {n}
        while True:
            new = set()
            for x in board:
                for i in range(1, n + 1):
                    if x % i == 1:
                        new.add(i)
            if new <= board:               # không thêm số nào: ổn định
                return len(board)
            board |= new

In [8]:
sol = SolutionSimulate()
print(sol.distinctIntegers(5), sol.distinctIntegers(3))    # 4 2
print([sol.distinctIntegers(n) for n in range(1, 9)])

4 2
[1, 1, 2, 3, 4, 5, 6, 7]


**Độ phức tạp:** Time `O(D * n^2)` với `D` là số ngày đến khi ổn định (`D <= n`), tức tối đa `O(n^3)`; với `n <= 100` là `10^6`. Space `O(n)`. **Nút thắt:** lặp lại việc duyệt cả `board` và mọi `i` mỗi ngày dù chỉ có số mới mới cần xét.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — BFS thay cho mô phỏng theo ngày.** Số `x` chỉ cần được "mở rộng" **một lần** (các `i` từ `x` luôn như nhau). Dùng hàng đợi: lấy `x`, thêm các `i` chưa có, đẩy `i` vào hàng đợi.

**Bước 2 — Cấu trúc toán học.** `x % i == 1` ⇔ `i >= 2` và `i | (x - 1)`. Với `x = n` (và `n >= 3`), số `i = n - 1` luôn là ước của `n - 1` nên được thêm. Với `x = n - 1`: `i = n - 2` là ước của `n - 2` nên cũng thêm (khi `n - 2 >= 2`). Cứ lặp: mọi số `k` từ `n - 1` xuống `2` đều xuất hiện. Số `1` không bao giờ xuất hiện (vì `x % 1 = 0`). Số lượng: `n - 1` giá trị `{2, ..., n}`.

**Bước 3 — Trường hợp nhỏ.** `n = 1`: chỉ có `1` trên bảng (`1 % i` với `i = 1` là `0`), đáp án `1`. `n = 2`: `2 % 1 = 0`, `2 % 2 = 0` nên bảng `{2}`, đáp án `1 = n - 1`.

**Quy tắc:** *đáp án là `max(1, n - 1)`: `1` nếu `n = 1`, ngược lại `n - 1`.*

**Invariant:** nếu `k` có trên bảng và `k >= 3` thì `k - 1` cũng sẽ có trên bảng (vì `k % (k - 1) = 1`).

## 7. Các cách tiếp theo

### Cách 2 — BFS với `deque`

Mỗi số được xử lý đúng một lần.

In [9]:
from collections import deque

class SolutionBFS:
    def distinctIntegers(self, n: int) -> int:
        seen = {n}
        q = deque([n])
        while q:
            x = q.popleft()
            for i in range(1, n + 1):
                if x % i == 1 and i not in seen:
                    seen.add(i)
                    q.append(i)
        return len(seen)

In [10]:
sol = SolutionBFS()
print(sol.distinctIntegers(5), sol.distinctIntegers(3))    # 4 2
print([sol.distinctIntegers(n) for n in range(1, 9)])

4 2
[1, 1, 2, 3, 4, 5, 6, 7]


**Độ phức tạp:** Time `O(n^2)` (mỗi số trong tối đa `n` số xử lý một lần, mỗi lần duyệt `n` giá trị `i`), Space `O(n)`.

### Cách 3 — Lan truyền theo ước của `x - 1`, từ lớn xuống nhỏ

Vì mọi số mới `i` thỏa `i <= x - 1 < x`, các số mới luôn **nhỏ hơn** số sinh ra nó. Nên chỉ cần duyệt `x` từ `n` giảm xuống `2`: nếu `reach[x]` đúng thì đánh dấu `reach[i] = True` cho mọi ước `i >= 2` của `x - 1`. Mỗi `x` được xét sau khi mọi số lớn hơn sinh ra nó đã xử lý xong.

In [11]:
class SolutionDivisors:
    def distinctIntegers(self, n: int) -> int:
        reach = [False] * (n + 1)
        reach[n] = True
        for x in range(n, 1, -1):                       # x từ lớn xuống nhỏ
            if not reach[x]:
                continue
            for i in range(2, x):                       # i là ước >= 2 của x - 1
                if (x - 1) % i == 0:
                    reach[i] = True
        return sum(reach)

In [12]:
sol = SolutionDivisors()
print(sol.distinctIntegers(5), sol.distinctIntegers(3))    # 4 2
print([sol.distinctIntegers(n) for n in range(1, 9)])

4 2
[1, 1, 2, 3, 4, 5, 6, 7]


**Độ phức tạp:** Time `O(n^2)`, Space `O(n)`. Điểm hay: không cần tập hợp động, chỉ cần mảng đánh dấu và duyệt theo thứ tự giảm dần.

### Cách 4 — Công thức `max(1, n - 1)`

Kết quả của mục 6.

In [13]:
class SolutionFormula:
    def distinctIntegers(self, n: int) -> int:
        return max(1, n - 1)

In [14]:
sol = SolutionFormula()
print(sol.distinctIntegers(5), sol.distinctIntegers(3))    # 4 2
print([sol.distinctIntegers(n) for n in range(1, 9)])

4 2
[1, 1, 2, 3, 4, 5, 6, 7]


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 (BFS) với `n = 5`:

| Bước | Lấy `x` | Các `i` thỏa `x % i == 1` | `seen` sau | Hàng đợi |
|---|---|---|---|---|
| 0 | | | `{5}` | `[5]` |
| 1 | 5 | 2, 4 | `{5,2,4}` | `[2,4]` |
| 2 | 2 | không có | `{5,2,4}` | `[4]` |
| 3 | 4 | 3 | `{5,2,4,3}` | `[3]` |
| 4 | 3 | 2 (đã có) | `{5,2,4,3}` | `[]` |

Kết quả `4`.

In [15]:
def trace_bfs(n):
    from collections import deque
    print("n =", n)
    seen = {n}
    q = deque([n])
    while q:
        x = q.popleft()
        found = [i for i in range(1, n + 1) if x % i == 1]
        new = [i for i in found if i not in seen]
        seen.update(new)
        q.extend(new)
        print(f"  lấy x={x}: i thỏa x % i == 1 là {found}, mới {new}, seen={sorted(seen)}")
    print("kết quả:", len(seen))
    return len(seen)

trace_bfs(5)
trace_bfs(6)
trace_bfs(2)
trace_bfs(1)

n = 5
  lấy x=5: i thỏa x % i == 1 là [2, 4], mới [2, 4], seen=[2, 4, 5]
  lấy x=2: i thỏa x % i == 1 là [], mới [], seen=[2, 4, 5]
  lấy x=4: i thỏa x % i == 1 là [3], mới [3], seen=[2, 3, 4, 5]
  lấy x=3: i thỏa x % i == 1 là [2], mới [], seen=[2, 3, 4, 5]
kết quả: 4
n = 6
  lấy x=6: i thỏa x % i == 1 là [5], mới [5], seen=[5, 6]
  lấy x=5: i thỏa x % i == 1 là [2, 4], mới [2, 4], seen=[2, 4, 5, 6]
  lấy x=2: i thỏa x % i == 1 là [], mới [], seen=[2, 4, 5, 6]
  lấy x=4: i thỏa x % i == 1 là [3], mới [3], seen=[2, 3, 4, 5, 6]
  lấy x=3: i thỏa x % i == 1 là [2], mới [], seen=[2, 3, 4, 5, 6]
kết quả: 5
n = 2
  lấy x=2: i thỏa x % i == 1 là [], mới [], seen=[2]
kết quả: 1
n = 1
  lấy x=1: i thỏa x % i == 1 là [], mới [], seen=[1]
kết quả: 1


1

## 9. Code nộp LeetCode

Công thức `O(1)`:

In [16]:
class Solution:
    def distinctIntegers(self, n: int) -> int:
        return max(1, n - 1)

In [17]:
print(Solution().distinctIntegers(5))
print(Solution().distinctIntegers(3))
print(Solution().distinctIntegers(1))

4
2
1


## 10. Vì sao thuật toán đúng

1. **Ổn định.** Nếu một ngày không thêm số mới thì bảng giữ nguyên ngày sau (tập kết quả sinh ra từ cùng bảng là như nhau). Vậy sau `10^9` ngày, bảng bằng bảng ổn định đầu tiên. Mỗi ngày không ổn định thêm ít nhất một số trong `[1, n]` nên chỉ có tối đa `n` ngày như vậy: `10^9` ngày thừa sức.
2. **Giá trị `1` không xuất hiện.** Muốn thêm `1` cần `x % 1 == 1`, nhưng `x % 1 == 0` với mọi `x`. Vậy `1` chỉ có mặt khi `n = 1` (ngay từ đầu).
3. **Mọi số từ `2` đến `n` xuất hiện (khi `n >= 2`).** Quy nạp lùi: `n` có mặt. Nếu `k` có mặt và `k >= 3`, `k % (k - 1) = 1` và `1 <= k - 1 <= n` nên `k - 1` được thêm. Vậy `n, n-1, ..., 2` đều có mặt.
4. **Không có số nào khác.** Số thêm vào phải thuộc `[1, n]` và khác `1`, nên chỉ có thể nằm trong `{2, ..., n}`.
5. **Đếm.** `n >= 2`: có `n - 1` số. `n = 1`: chỉ số `1`, đáp án `1`. Gộp lại `max(1, n - 1)`. Với `n = 2`: bảng `{2}`, `2 % 2 = 0` nên không thêm gì, và công thức `max(1, 1) = 1` đúng.
6. **Các cách mô phỏng/BFS/ước** cùng tính tập hợp này nên cho cùng kết quả; test mục 12 kiểm chứng điều đó cho mọi `n = 1..100`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng ngày | `O(n^3)` (tối đa `n` ngày, mỗi ngày `n^2`) | `O(n)` |
| 2. BFS | `O(n^2)` | `O(n)` |
| 3. Lan truyền theo ước | `O(n^2)` | `O(n)` |
| 4. Công thức | `O(1)` | `O(1)` |

`n <= 100` nên mọi cách đều rất nhanh; công thức cho đáp án trực tiếp.

## 12. Bộ test

### 12.1. Ví dụ của đề, edge case và phản ví dụ

Mỗi kỳ vọng đều tính tay (ghi chú bên cạnh).

In [18]:
sols = [SolutionSimulate(), SolutionBFS(), SolutionDivisors(), SolutionFormula(), Solution()]
cases = [
    (5, 4),        # ví dụ 1: {2,3,4,5}
    (3, 2),        # ví dụ 2: {2,3}
    (1, 1),        # chỉ có 1
    (2, 1),        # 2 % 1 = 0, 2 % 2 = 0 nên {2}
    (4, 3),        # {4,3,2}
    (6, 5),        # {6,5,4,3,2}
    (7, 6),        # {7,6,5,4,3,2}
    (10, 9),
    (99, 98),
    (100, 99),     # biên trên
]
for n, want in cases:
    for s in sols:
        got = s.distinctIntegers(n)
        assert got == want, (type(s).__name__, n, got, want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 10


Kiểm tra tay `n = 4`: ngày 1: `4 % 3 = 1` nên thêm `3` (còn `4 % 2 = 0`, `4 % 4 = 0`); ngày 2: `3 % 2 = 1` nên thêm `2`; ngày 3: `2` không sinh gì. Bảng `{4, 3, 2}` gồm 3 số, bằng `4 - 1`.

### 12.2. Đối chiếu toàn bộ miền và ngẫu nhiên (có seed)

Miền `n = 1..100` chỉ có 100 giá trị nên kiểm tra **tất cả**; thêm một lượt ngẫu nhiên có seed cho đủ yêu cầu.

In [19]:
import random
random.seed(2549)

for n in range(1, 101):
    ans = [s.distinctIntegers(n) for s in sols]
    assert len(set(ans)) == 1, (n, ans)

for _ in range(1000):
    n = random.randint(1, 100)
    ans = [s.distinctIntegers(n) for s in sols]
    assert len(set(ans)) == 1 and ans[0] == max(1, n - 1), (n, ans)
print("toàn bộ n = 1..100 và 1000 test ngẫu nhiên: 5 cách khớp nhau")

toàn bộ n = 1..100 và 1000 test ngẫu nhiên: 5 cách khớp nhau


### 12.3. Kiểm tra "ổn định" và "không có số 1"

Dùng mô phỏng để xác nhận hai khẳng định ở mục 10: bảng ổn định đạt được sau không quá `n` ngày, và số `1` chỉ xuất hiện khi `n = 1`.

In [20]:
def simulate_days(n):
    board = {n}
    days = 0
    while True:
        new = {i for x in board for i in range(1, n + 1) if x % i == 1}
        if new <= board:
            return board, days
        board |= new
        days += 1

for n in range(1, 101):
    board, days = simulate_days(n)
    assert days <= n
    assert (1 in board) == (n == 1)
    assert board == (set(range(2, n + 1)) if n >= 2 else {1})
print("mọi n trong 1..100: ổn định sau <= n ngày; bảng cuối là {2..n} (hoặc {1} khi n = 1)")
print("số ngày để ổn định với n = 100:", simulate_days(100)[1])

mọi n trong 1..100: ổn định sau <= n ngày; bảng cuối là {2..n} (hoặc {1} khi n = 1)
số ngày để ổn định với n = 100: 50


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ mô phỏng **một ngày**

In [21]:
def sai_mot_ngay(n):
    board = {n}
    board |= {i for i in range(1, n + 1) if n % i == 1}       # SAI: chỉ ngày đầu tiên
    return len(board)

print("n=5 đúng:", Solution().distinctIntegers(5), "| sai:", sai_mot_ngay(5))
assert Solution().distinctIntegers(5) != sai_mot_ngay(5)

n=5 đúng: 4 | sai: 3


**Vì sao sai:** đề lặp qua `10^9` ngày; các số mới (như `4`) lại sinh thêm số khác (`3`) ở ngày sau. Một ngày chỉ cho `{5, 2, 4}`.

### Lỗi 2: Trả về `n` (tưởng rằng bảng chỉ chứa số ban đầu hoặc mọi số)

In [22]:
def sai_tra_n(n):
    return n                                                   # SAI: số 1 không bao giờ có mặt

print("n=5 đúng:", Solution().distinctIntegers(5), "| sai:", sai_tra_n(5))
assert Solution().distinctIntegers(5) != sai_tra_n(5)

n=5 đúng: 4 | sai: 5


**Vì sao sai:** số `1` không bao giờ xuất hiện (vì `x % 1 == 0`), nên bảng cuối có `n - 1` số `{2, ..., n}`, không phải `n`.

### Lỗi 3: Trả về `n - 1` cho mọi `n`, quên trường hợp `n = 1`

In [23]:
def sai_n_tru_1(n):
    return n - 1                                               # SAI khi n = 1

print("n=1 đúng:", Solution().distinctIntegers(1), "| sai:", sai_n_tru_1(1))
assert Solution().distinctIntegers(1) != sai_n_tru_1(1)

n=1 đúng: 1 | sai: 0


**Vì sao sai:** với `n = 1`, bảng ban đầu đã có số `1` và không thêm gì, nên đáp án `1`. Công thức `n - 1 = 0` vô lý vì bảng không thể rỗng.

### Lỗi 4: Nhầm `x % i == 1` với `x % i == 0` (tìm ước thay vì dư 1)

In [24]:
def sai_uoc(n):
    board = {n}
    while True:
        new = {i for x in board for i in range(1, n + 1) if x % i == 0}      # SAI: dư 0 (ước), không phải dư 1
        if new <= board:
            return len(board)
        board |= new

print("n=5 đúng:", Solution().distinctIntegers(5), "| sai:", sai_uoc(5))
assert Solution().distinctIntegers(5) != sai_uoc(5)

n=5 đúng: 4 | sai: 2


**Vì sao sai:** đề yêu cầu số dư bằng `1`. Dùng `== 0` sinh ra các **ước** của `x`, ví dụ từ `5` chỉ có `{1, 5}`, đếm được `2`.

### Lỗi 5: Dùng `list` thay `set`, đếm trùng

In [25]:
def sai_list(n):
    board = [n]
    for _ in range(5):                                        # vài ngày là đủ để thấy trùng lặp
        for x in list(board):
            for i in range(1, n + 1):
                if x % i == 1:
                    board.append(i)                            # SAI: không loại trùng
    return len(board)

print("n=5 đúng:", Solution().distinctIntegers(5), "| sai:", sai_list(5))
assert Solution().distinctIntegers(5) != sai_list(5)

n=5 đúng: 4 | sai: 31


**Vì sao sai:** đề yêu cầu số lượng giá trị **khác nhau**. Dùng `list` không loại trùng cho số phần tử rất lớn; hãy dùng `set` (hoặc mảng đánh dấu).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "lặp `10^9` ngày" ⇒ khả năng cao quy trình **ổn định rất nhanh**; tìm trạng thái bất biến thay vì mô phỏng thật.
- Với bài "thêm số mới sinh ra từ số cũ" ⇒ **BFS** hoặc đánh dấu lan truyền.
- Điều kiện `x % i == 1` ⇒ `i` là ước của `x - 1` (và `i >= 2`).
- Hãy **tính tay vài `n` nhỏ** rồi đoán quy luật: `n = 1..6` cho `1, 1, 2, 3, 4, 5`.
- Nhớ ngoại lệ `n = 1` (đáp án `1`) khi dùng công thức `n - 1`.
- Dùng `max(1, n - 1)` để gộp hai trường hợp.

In [26]:
print([max(1, n - 1) for n in range(1, 11)])

[1, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 15. Biến thể của bài

1. **Bảng khởi đầu nhiều số:** ban đầu có tập `S`; hỏi bảng cuối (hợp các bảng sinh ra từ từng số).
2. **Đổi điều kiện:** thêm `i` thỏa `x % i == 2`, hoặc `x % i == r` tổng quát.
3. **Giới hạn số ngày `d`:** đếm số khác nhau sau đúng `d` ngày (cần BFS theo tầng).
4. **Giới hạn `i` lớn hơn `n`:** thay `1 <= i <= n` bằng `1 <= i <= M`.
5. **Số ngày ít nhất để bảng ổn định:** bài toán độ sâu BFS (với `n` lớn là khoảng `n - 2` ngày).
6. **Các bài BFS trên số nguyên:** biến đổi `x` thành `y` theo một quy tắc cho trước và đếm số trạng thái đạt được.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng ngày | lặp đến khi bảng không đổi | `O(n^3)` | `O(n)` | hiểu đề, làm oracle |
| 2. BFS | mỗi số xử lý một lần | `O(n^2)` | `O(n)` | khi chưa biết công thức |
| 3. Lan truyền theo ước | duyệt từ lớn xuống nhỏ, ước của `x - 1` | `O(n^2)` | `O(n)` | khi muốn dùng tính chất số học |
| 4. Công thức | `max(1, n - 1)` | `O(1)` | `O(1)` | cách nộp bài chuẩn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().distinctIntegers(8)` bằng bao nhiêu? Còn `Solution().distinctIntegers(2)`?

**Bài 2 (tính tay).** Với `n = 6`, liệt kê các số được thêm vào mỗi ngày.

**Bài 3 (điền TODO).** Hoàn thành `days_to_stabilize(n)` trả về số ngày **có thêm số mới** cho đến khi bảng ổn định.

In [27]:
def days_to_stabilize(n):
    board = {n}
    days = 0
    # TODO: lặp: tính new = {i : x % i == 1}; nếu new <= board thì dừng; ngược lại board |= new, days += 1
    return days

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `distinct_with_remainder(n, r)`: giống đề nhưng điều kiện là `x % i == r` (với `r >= 0`), `1 <= i <= n`, dùng BFS. Kiểm tra `distinct_with_remainder(n, 1) == Solution().distinctIntegers(n)`.

**Bài 5 (biến thể / mini challenge).** Viết `distinct_from_set(start, n)`: bảng ban đầu là **tập** `start` (các số trong `[1, n]`), áp dụng đề; trả về số lượng giá trị khác nhau ở bảng cuối.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `max(1, 8 - 1) = 7`; `n = 2` ra `1`.
- **Mức 2 (vừa):** Bài 2: `n = 6`: ngày 1 từ `6`: `6 % 5 = 1` nên thêm `5` (còn `6 % 1,2,3,4,6`: `0, 0, 0, 2, 0`); ngày 2 từ `5`: thêm `2` và `4`; ngày 3 từ `4`: thêm `3`; ngày 4 không thêm. Bài 3: đếm các vòng lặp có thêm số mới.
- **Mức 3 (sâu):** Bài 4: BFS giống Cách 2 nhưng đổi `== 1` thành `== r`; với `r = 1` trùng đáp án đề. Bài 5: khởi tạo `seen = set(start)` và `q = deque(start)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from collections import deque

# Bài 1
assert Solution().distinctIntegers(8) == 7
assert Solution().distinctIntegers(2) == 1

# Bài 2: n = 6
assert simulate_days(6)[0] == {2, 3, 4, 5, 6}

# Bài 3
def days_to_stabilize(n):
    board = {n}
    days = 0
    while True:
        new = {i for x in board for i in range(1, n + 1) if x % i == 1}
        if new <= board:
            return days
        board |= new
        days += 1

# Bài 4
def distinct_with_remainder(n, r):
    seen = {n}
    q = deque([n])
    while q:
        x = q.popleft()
        for i in range(1, n + 1):
            if x % i == r and i not in seen:
                seen.add(i)
                q.append(i)
    return len(seen)

# Bài 5
def distinct_from_set(start, n):
    seen = set(start)
    q = deque(start)
    while q:
        x = q.popleft()
        for i in range(1, n + 1):
            if x % i == 1 and i not in seen:
                seen.add(i)
                q.append(i)
    return len(seen)

assert days_to_stabilize(1) == 0
assert days_to_stabilize(2) == 0
assert days_to_stabilize(5) == 2      # ngày 1 thêm {2,4}, ngày 2 thêm {3}
assert days_to_stabilize(6) == 3
assert all(days_to_stabilize(n) == simulate_days(n)[1] for n in range(1, 60))
assert all(distinct_with_remainder(n, 1) == Solution().distinctIntegers(n) for n in range(1, 101))
assert distinct_with_remainder(5, 0) == 2
assert distinct_from_set({5}, 5) == 4
assert distinct_from_set({2}, 5) == 1
assert distinct_from_set({2, 5}, 5) == 4
random.seed(49)
for _ in range(200):
    n = random.randint(1, 100)
    assert distinct_from_set({n}, n) == Solution().distinctIntegers(n)
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tìm trạng thái ổn định rồi rút ra công thức)

```python
class Solution:
    def distinctIntegers(self, n: int) -> int:
        return max(1, n - 1)       # bảng cuối: {2, ..., n} (hoặc {1} khi n = 1)
```

**Checklist**

- [ ] Hiểu rằng `10^9` ngày chỉ là cách nói "đến khi ổn định"?
- [ ] Nhớ `x % 1 == 0` nên số `1` không bao giờ được thêm?
- [ ] Đáp án `n - 1` cho `n >= 2` và `1` cho `n = 1` (`max(1, n - 1)`)?
- [ ] Đã thử tay `n = 1, 2, 3, 4, 5` và khớp ví dụ của đề (`5 -> 4`, `3 -> 2`)?
- [ ] Dùng `set` (không phải `list`) khi đếm số khác nhau?
- [ ] Điều kiện là dư `== 1` (không phải `== 0`)?
- [ ] Signature đúng `distinctIntegers(self, n: int) -> int`?